In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

### This code loads the Adult dataset and assigns column names.

In [ ]:
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)
df.head()

### This code checks the dataset shape, column types, and non-null values.

In [ ]:
print("Shape:", df.shape)
df.info()

### This code shows descriptive statistics for numerical columns.

In [ ]:
df.describe()

### This code replaces '?' values with NaN so missing values can be handled properly.

In [ ]:
df.replace("?", np.nan, inplace=True)
df.isnull().sum()

### This code checks the target variable distribution.

In [ ]:
df["income"].value_counts()

### This code visualizes the income target variable distribution.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="income", data=df)
plt.title("Income Distribution")
plt.show()

### This code visualizes income distribution by sex.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="income", hue="sex", data=df)
plt.title("Income by Sex")
plt.show()

### This code visualizes the age distribution.

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["age"], bins=30)
plt.title("Age Distribution")
plt.show()

### This code creates predictors X and target y.

In [ ]:
X = df.drop("income", axis=1)
y = df["income"]

X.head()

### This code identifies categorical and numerical columns for preprocessing.

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numeric_cols)

### This code splits the dataset into training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### This code builds the preprocessing pipeline for categorical and numerical features.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols)
    ]
)

preprocessor

### This code creates and trains a Decision Tree classifier.

In [ ]:
dtree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

dtree.fit(X_train, y_train)

### This code generates predictions using the Decision Tree model.

In [ ]:
dtree_predictions = dtree.predict(X_test)
dtree_predictions[:10]

### This code evaluates the Decision Tree model using accuracy, confusion matrix, and classification report.

In [ ]:
print("Decision Tree Accuracy:", accuracy_score(y_test, dtree_predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dtree_predictions))
print("\nClassification Report:")
print(classification_report(y_test, dtree_predictions))

### This code visualizes the Decision Tree confusion matrix.

In [ ]:
dtree_cm = confusion_matrix(y_test, dtree_predictions, labels=dtree.named_steps["classifier"].classes_)

plt.figure(figsize=(6,4))
sns.heatmap(
    dtree_cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=dtree.named_steps["classifier"].classes_,
    yticklabels=dtree.named_steps["classifier"].classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Decision Tree Confusion Matrix")
plt.show()

### This code creates and trains a Random Forest classifier.

In [ ]:
rfc = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])

rfc.fit(X_train, y_train)

### This code generates predictions using the Random Forest model.

In [ ]:
rfc_predictions = rfc.predict(X_test)
rfc_predictions[:10]

### This code evaluates the Random Forest model using accuracy, confusion matrix, and classification report.

In [ ]:
print("Random Forest Accuracy:", accuracy_score(y_test, rfc_predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rfc_predictions))
print("\nClassification Report:")
print(classification_report(y_test, rfc_predictions))

### This code visualizes the Random Forest confusion matrix.

In [ ]:
rfc_cm = confusion_matrix(y_test, rfc_predictions, labels=rfc.named_steps["classifier"].classes_)

plt.figure(figsize=(6,4))
sns.heatmap(
    rfc_cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=rfc.named_steps["classifier"].classes_,
    yticklabels=rfc.named_steps["classifier"].classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

### This code compares Decision Tree and Random Forest accuracy.

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, dtree_predictions),
        accuracy_score(y_test, rfc_predictions)
    ]
})

model_comparison

### This code visualizes the model accuracy comparison.

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x="Model", y="Accuracy", data=model_comparison)
plt.ylim(0, 1)
plt.title("Decision Tree vs Random Forest Accuracy")
plt.show()

### This code extracts feature names after one-hot encoding.

In [ ]:
encoder = rfc.named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]
encoded_cat_features = encoder.get_feature_names_out(categorical_cols)

all_feature_names = np.concatenate([encoded_cat_features, np.array(numeric_cols)])
all_feature_names[:10]

### This code gets feature importance from the Random Forest model.

In [ ]:
feature_importances = rfc.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": feature_importances
}).sort_values(by="Importance", ascending=False)

importance_df.head(15)

### This code visualizes the top 15 most important features from the Random Forest model.

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x="Importance", y="Feature", data=importance_df.head(15))
plt.title("Top 15 Random Forest Feature Importances")
plt.show()